# Phase 1: Project Ingest & Reconciliation

One-stop workflow for ingesting project skills into central repo.

1. **Scan** — compare project vs. central
2. **Classify** — every new skill (Tracked/Fork/Own/Skip)
3. **Reconcile** — changed skills (Central/Project/Skip)
4. **Apply** — import & update manifests
5. **Verify** — ledger self-check
6. **Next** — ready for CSV adjudication

In [ ]:
# Setup
import json
import shutil
import difflib
from pathlib import Path
from typing import Optional, Dict, List
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# Utilities
def find_repo_root(start: Path | None = None) -> Path:
    start = start or Path.cwd()
    for p in [start, *start.parents]:
        if (p / "manifests" / "origins.json").is_file():
            return p
    raise FileNotFoundError("manifests/origins.json not found")

def scan_skills_directory(path: Path) -> dict[str, Path]:
    skills = {}
    if not path.is_dir():
        return skills
    for skill_dir in path.iterdir():
        if skill_dir.is_dir() and (skill_dir / "SKILL.md").is_file():
            skills[skill_dir.name] = skill_dir
    return skills

def compare_skills(repo_skills: dict[str, Path], project_skills: dict[str, Path]) -> dict:
    repo_names = set(repo_skills.keys())
    project_names = set(project_skills.keys())
    return {
        "new": sorted(project_names - repo_names),
        "existing": sorted(project_names & repo_names),
        "removed": sorted(repo_names - project_names),
    }

def check_skill_changed(repo_path: Path, project_path: Path) -> bool:
    repo_md = (repo_path / "SKILL.md").read_bytes()
    project_md = (project_path / "SKILL.md").read_bytes()
    return repo_md != project_md

def show_diff(repo_path: Path, project_path: Path, skill_name: str) -> str:
    repo_lines = (repo_path / "SKILL.md").read_text(encoding="utf-8").splitlines(keepends=True)
    project_lines = (project_path / "SKILL.md").read_text(encoding="utf-8").splitlines(keepends=True)
    diff = list(difflib.unified_diff(
        repo_lines, project_lines,
        fromfile=f"central/{skill_name}/SKILL.md",
        tofile=f"project/{skill_name}/SKILL.md",
        n=1
    ))
    lines = ["".join(diff[:30])]
    if len(diff) > 30:
        lines.append(f"... ({len(diff) - 30} more lines)")
    return "".join(lines)

# Global state
repo_root = find_repo_root()
skills_dir = repo_root / "skills"
print(f"✓ Repo root: {repo_root}")
print(f"✓ Skills dir: {skills_dir}")

## Phase 1a: Scan & Dry-Run

In [ ]:
# Input widgets
source_path_input = widgets.Text(
    value="",
    placeholder="C:/path/to/project/.agents/skills",
    description="Source path:",
    style={"description_width": "150px"},
    layout=widgets.Layout(width="600px")
)

destination_id_input = widgets.Text(
    value="campus-profile",
    placeholder="campus-profile",
    description="Destination ID:",
    style={"description_width": "150px"},
    layout=widgets.Layout(width="300px")
)

scan_output = widgets.Output()
scan_button = widgets.Button(description="Scan", button_style="info")

def on_scan_clicked(b):
    scan_output.clear_output()
    with scan_output:
        source = Path(source_path_input.value).expanduser()
        dest_id = destination_id_input.value.strip()
        
        if not source.is_dir():
            print(f"❌ Source not found: {source}")
            return
        if not dest_id:
            print(f"❌ Destination ID required")
            return
        
        # Scan
        repo_skills = scan_skills_directory(skills_dir)
        project_skills = scan_skills_directory(source)
        diff = compare_skills(repo_skills, project_skills)
        
        # Store for later phases
        global current_scan
        current_scan = {
            "source": source,
            "dest_id": dest_id,
            "repo_skills": repo_skills,
            "project_skills": project_skills,
            "diff": diff,
        }
        
        # Report
        print(f"{'='*70}")
        print(f"PHASE 1a: Initial Intake & Reconciliation")
        print(f"Project: {dest_id}")
        print(f"{'='*70}")
        print(f"Source:  {source}")
        print(f"Central: {skills_dir}\n")
        
        if diff["new"]:
            print(f"✚ NEW SKILLS ({len(diff['new'])})")
            for name in diff["new"]:
                print(f"    {name}")
            print()
        
        changed = []
        unchanged = []
        if diff["existing"]:
            for name in diff["existing"]:
                if check_skill_changed(repo_skills[name], project_skills[name]):
                    changed.append(name)
                else:
                    unchanged.append(name)
            
            if changed:
                print(f"~ CHANGED SKILLS ({len(changed)})")
                for name in changed:
                    print(f"    {name}")
                print()
            
            if unchanged:
                print(f"= UNCHANGED SKILLS ({len(unchanged)})")
                for name in unchanged[:5]:
                    print(f"    {name}")
                if len(unchanged) > 5:
                    print(f"    ... and {len(unchanged) - 5} more")
                print()
        
        if diff["removed"]:
            print(f"- NOT IN PROJECT ({len(diff['removed'])})")
            for name in diff["removed"][:5]:
                print(f"    {name}")
            if len(diff["removed"]) > 5:
                print(f"    ... and {len(diff['removed']) - 5} more")
            print()
        
        # Store changed for reconciliation
        current_scan["changed"] = changed
        current_scan["unchanged"] = unchanged
        
        print(f"{'='*70}")
        print("✓ Scan complete. Ready for Phase 1b (Classification).")

scan_button.on_click(on_scan_clicked)

display(widgets.VBox([
    source_path_input,
    destination_id_input,
    scan_button,
    scan_output
]))

## Phase 1b: Classify New Skills

In [ ]:
# Classification UI for each new skill
classify_output = widgets.Output()
classifications = {}  # {skill_name: classification_dict}

def build_classify_widgets():
    if "current_scan" not in globals() or not current_scan["diff"]["new"]:
        classify_output.clear_output()
        with classify_output:
            print("⚠️  No new skills to classify. Run Scan first.")
        return
    
    classify_output.clear_output()
    with classify_output:
        print(f"{'='*70}")
        print(f"PHASE 1b: Classify {len(current_scan['diff']['new'])} New Skill(s)")
        print(f"{'='*70}\n")
        
        # Build a widget for each new skill
        for skill_name in current_scan["diff"]["new"]:
            print(f"\n🔹 {skill_name}\n")
            
            # Classification choice
            choice_radio = widgets.RadioButtons(
                options=[
                    ("[T] Tracked upstream (auto-updated)", "tracked"),
                    ("[F] Fork (modified from upstream)", "fork"),
                    ("[O] Own (no upstream source)", "own"),
                    ("[S] Skip (don't import)", "skip"),
                ],
                layout=widgets.Layout(width="400px")
            )
            
            # Conditional fields
            repo_field = widgets.Text(
                placeholder="https://github.com/...",
                description="Repo:",
                style={"description_width": "80px"},
                visible=False
            )
            branch_field = widgets.Text(
                value="main",
                description="Branch:",
                style={"description_width": "80px"},
                visible=False
            )
            subpath_field = widgets.Text(
                placeholder=f".agents/skills/{skill_name}",
                description="Subpath:",
                style={"description_width": "80px"},
                visible=False
            )
            author_field = widgets.Text(
                placeholder="Author",
                description="Author:",
                style={"description_width": "80px"},
                visible=False
            )
            origin_field = widgets.Text(
                placeholder="e.g., microsoft/fabric-apps",
                description="Origin:",
                style={"description_width": "80px"},
                visible=False
            )
            own_field = widgets.Text(
                placeholder="Owner/author",
                description="Owner:",
                style={"description_width": "80px"},
                visible=False
            )
            
            def on_choice_change(change, sn=skill_name):
                choice = change["new"]
                repo_field.visible = choice == "tracked"
                branch_field.visible = choice == "tracked"
                subpath_field.visible = choice == "tracked"
                author_field.visible = choice == "tracked"
                origin_field.visible = choice == "fork"
                own_field.visible = choice == "own"
            
            choice_radio.observe(on_choice_change, names="value")
            
            # Store for later
            classifications[skill_name] = {
                "choice": choice_radio,
                "repo": repo_field,
                "branch": branch_field,
                "subpath": subpath_field,
                "author": author_field,
                "origin": origin_field,
                "owner": own_field,
            }
            
            display(choice_radio)
            display(widgets.VBox([
                repo_field, branch_field, subpath_field, author_field,
                origin_field, own_field
            ]))

build_button = widgets.Button(description="Build Classification Widgets", button_style="warning")
build_button.on_click(lambda b: build_classify_widgets())

display(build_button)
display(classify_output)

## Phase 1c: Reconcile Changed Skills

In [ ]:
reconcile_output = widgets.Output()
reconciliations = {}  # {skill_name: choice}

def build_reconcile_widgets():
    if "current_scan" not in globals() or not current_scan.get("changed"):
        reconcile_output.clear_output()
        with reconcile_output:
            print("ℹ️  No changed skills to reconcile.")
        return
    
    reconcile_output.clear_output()
    with reconcile_output:
        print(f"{'='*70}")
        print(f"PHASE 1c: Reconcile {len(current_scan['changed'])} Changed Skill(s)")
        print(f"{'='*70}\n")
        
        for skill_name in current_scan["changed"]:
            repo_path = current_scan["repo_skills"][skill_name]
            project_path = current_scan["project_skills"][skill_name]
            
            print(f"\n🔹 {skill_name}")
            print(f"\n{show_diff(repo_path, project_path, skill_name)}\n")
            
            choice_radio = widgets.RadioButtons(
                options=[
                    ("[C] Keep central version (project drifted)", "central"),
                    ("[P] Take project version (intentional edits)", "project"),
                    ("[S] Skip (leave as-is)", "skip"),
                ],
                layout=widgets.Layout(width="400px")
            )
            
            reconciliations[skill_name] = choice_radio
            display(choice_radio)

reconcile_button = widgets.Button(description="Build Reconciliation Widgets", button_style="warning")
reconcile_button.on_click(lambda b: build_reconcile_widgets())

display(reconcile_button)
display(reconcile_output)

## Phase 1d: Apply Changes

In [ ]:
apply_output = widgets.Output()
target_dir_input = widgets.Text(
    value="",
    placeholder="C:/path/to/project/.agents/skills",
    description="Target dir:",
    style={"description_width": "150px"},
    layout=widgets.Layout(width="600px")
)

apply_button = widgets.Button(description="Apply Changes", button_style="success")

def on_apply_clicked(b):
    apply_output.clear_output()
    with apply_output:
        if "current_scan" not in globals():
            print("❌ No scan in progress. Run Scan first.")
            return
        
        target_dir = Path(target_dir_input.value).expanduser()
        if not target_dir.is_dir() and target_dir != Path("."):
            print(f"⚠️  Target dir will be created: {target_dir}")
        
        print(f"{'='*70}")
        print(f"PHASE 1d: Applying Changes")
        print(f"{'='*70}\n")
        
        # Load manifests
        origins_path = repo_root / "manifests" / "origins.json"
        destinations_path = repo_root / "manifests" / "destinations.json"
        origins = json.loads(origins_path.read_text(encoding="utf-8"))
        destinations = json.loads(destinations_path.read_text(encoding="utf-8"))
        
        skills_assigned = list(current_scan["unchanged"])
        imported_new = 0
        imported_changed = 0
        
        # Import new skills
        print(f"--- Importing {len(current_scan['diff']['new'])} new skill(s) ---")
        for skill_name in current_scan["diff"]["new"]:
            if skill_name not in classifications:
                print(f"⚠️  {skill_name}: not classified, skipping")
                continue
            
            choice = classifications[skill_name]["choice"].value
            
            if choice == "skip":
                print(f"⊘ {skill_name}: skipped")
                continue
            
            # Copy to central
            src = current_scan["project_skills"][skill_name]
            dst = skills_dir / skill_name
            shutil.copytree(src, dst, dirs_exist_ok=True)
            imported_new += 1
            skills_assigned.append(skill_name)
            
            if choice == "tracked":
                if not any(s["name"] == skill_name for s in origins["skills"]):
                    origins["skills"].append({
                        "name": skill_name,
                        "local": f"skills/{skill_name}",
                        "repo": classifications[skill_name]["repo"].value,
                        "branch": classifications[skill_name]["branch"].value,
                        "subpath": classifications[skill_name]["subpath"].value,
                        "type": "skill",
                        "format": "skill-folder",
                        "author": classifications[skill_name]["author"].value,
                        "license": "",
                        "last_synced_sha": None,
                    })
                print(f"✓ {skill_name} → tracked")
            
            elif choice in ("fork", "own"):
                if not any(e["name"] == skill_name for e in origins["excluded"]):
                    if choice == "fork":
                        reason = f"Fork of {classifications[skill_name]['origin'].value}; locally modified, do not auto-update."
                    else:
                        owner = classifications[skill_name]["owner"].value or "Project"
                        reason = f"Own skill ({owner}); no upstream source."
                    
                    origins["excluded"].append({
                        "name": skill_name,
                        "local": f"skills/{skill_name}",
                        "reason": reason,
                    })
                label = "fork" if choice == "fork" else "own"
                print(f"✓ {skill_name} → {label}")
        
        # Reconcile changed skills
        if current_scan["changed"]:
            print(f"\n--- Reconciling {len(current_scan['changed'])} changed skill(s) ---")
            for skill_name in current_scan["changed"]:
                if skill_name not in reconciliations:
                    print(f"⚠️  {skill_name}: not reconciled, skipping")
                    continue
                
                choice = reconciliations[skill_name].value
                
                if choice == "central":
                    skills_assigned.append(skill_name)
                    print(f"✓ {skill_name} → keeping central version")
                
                elif choice == "project":
                    src = current_scan["project_skills"][skill_name]
                    dst = current_scan["repo_skills"][skill_name]
                    shutil.rmtree(dst)
                    shutil.copytree(src, dst)
                    skills_assigned.append(skill_name)
                    imported_changed += 1
                    print(f"✓ {skill_name} → updated from project")
                
                elif choice == "skip":
                    print(f"⊘ {skill_name}: skipped")
        
        # Write manifests
        origins_path.write_text(
            json.dumps(origins, indent=2, ensure_ascii=False) + "\n",
            encoding="utf-8"
        )
        
        # Add destination
        new_dest = {
            "id": current_scan["dest_id"],
            "environment": current_scan["dest_id"].replace("_", "-"),
            "type": "skill-folder",
            "method": "skill-folder-copy",
            "target_dir": str(target_dir),
            "skills_assigned": sorted(skills_assigned),
            "enabled": False,
        }
        destinations["destinations"].append(new_dest)
        destinations_path.write_text(
            json.dumps(destinations, indent=2, ensure_ascii=False) + "\n",
            encoding="utf-8"
        )
        
        print(f"\n{'='*70}")
        print(f"✓ PHASE 1d: Changes Applied")
        print(f"{'='*70}")
        print(f"Imported: {imported_new} new + {imported_changed} changed")
        print(f"Assigned: {len(skills_assigned)} skills to {current_scan['dest_id']}")
        print(f"Destination: {current_scan['dest_id']} (DISABLED)")
        print(f"\nManifests updated:")
        print(f"  ✓ origins.json")
        print(f"  ✓ destinations.json")
        
        # Mark for next phase
        global phase1_complete
        phase1_complete = True

apply_button.on_click(on_apply_clicked)

display(widgets.VBox([
    target_dir_input,
    apply_button,
    apply_output
]))

## Phase 1e: Verify Ledger

In [ ]:
verify_output = widgets.Output()
verify_button = widgets.Button(description="Verify Skills Ledger", button_style="info")

def on_verify_clicked(b):
    verify_output.clear_output()
    with verify_output:
        if "phase1_complete" not in globals():
            print("⚠️  Phase 1d not complete. Run Apply first.")
            return
        
        print(f"{'='*70}")
        print(f"PHASE 1e: Verify Skills Ledger")
        print(f"{'='*70}\n")
        
        # Reload manifests
        origins_path = repo_root / "manifests" / "origins.json"
        origins = json.loads(origins_path.read_text(encoding="utf-8"))
        
        # Scan skills/ directory
        on_disk = set()
        for skill_dir in skills_dir.iterdir():
            if skill_dir.is_dir() and (skill_dir / "SKILL.md").is_file():
                on_disk.add(skill_dir.name)
        
        # Check manifests
        in_manifest = set()
        for skill in origins.get("skills", []):
            in_manifest.add(skill["name"])
        for skill in origins.get("excluded", []):
            in_manifest.add(skill["name"])
        
        # Compare
        orphaned = on_disk - in_manifest
        missing = in_manifest - on_disk
        
        if not orphaned and not missing:
            print(f"✓ Ledger clean: {len(on_disk)} skills on disk, all in origins.json")
            print(f"\n{'='*70}")
            print("Ready for Phase 2: CSV Adjudication")
            print(f"{'='*70}")
            print(f"\nNext steps:")
            print(f"  1. py scripts/generate-destinations-csv.py")
            print(f"  2. Edit destinations-matrix.csv")
            print(f"  3. py scripts/update-destinations-from-csv.py")
            print(f"  4. Run sync_orchestrator.ipynb")
        else:
            if orphaned:
                print(f"❌ Orphaned skills (on disk, not in manifest):")
                for name in sorted(orphaned):
                    print(f"    {name}")
            if missing:
                print(f"❌ Missing skills (in manifest, not on disk):")
                for name in sorted(missing):
                    print(f"    {name}")
            print(f"\n⚠️  Fix manifest inconsistencies before Phase 2.")

verify_button.on_click(on_verify_clicked)

display(widgets.VBox([
    verify_button,
    verify_output
]))

## Summary

After Phase 1e passes verification, proceed to **Phase 2: CSV Adjudication**

Run in terminal:
```bash
py scripts/generate-destinations-csv.py
# Edit destinations-matrix.csv to assign skills to destinations
py scripts/update-destinations-from-csv.py
```

Then run `sync_orchestrator.ipynb` to distribute skills.